# 4. Hyperparameter Tuning

RandomizedSearchCV for LightGBM and XGBoost using walk-forward CV folds.

**Baseline (v3_macro default params):**
- Ensemble holdout MAE: $26,044
- XGBoost holdout MAE: $25,786
- LightGBM holdout MAE: $27,600

**Goal:** Find better hyperparameters to reduce MAE, then re-evaluate ensemble.

In [11]:
%pip install scikit-learn lightgbm xgboost --quiet

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error, mean_absolute_percentage_error
from scipy.stats import randint, uniform
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

## 1. Load Data & Setup

In [13]:
df_train = pd.read_csv("../data/processed/train.csv", low_memory=False)
df_test = pd.read_csv("../data/processed/test.csv", low_memory=False)

TARGET = "resale_price"
y_train = df_train[TARGET]
y_test = df_test[TARGET]
X_train = df_train.drop(columns=[TARGET])
X_test = df_test.drop(columns=[TARGET])

cat_cols = ["town", "flat_type", "flat_model", "closest_mrt", "nearest_carpark_type"]
bool_cols = [c for c in X_train.columns if X_train[c].dtype == "bool"]
num_cols = [c for c in X_train.columns if c not in cat_cols + bool_cols]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("bool", "passthrough", bool_cols),
])

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print(f"Train: {X_train_t.shape}, Test: {X_test_t.shape}")

Train: (667327, 230), Test: (18827, 230)


In [14]:
# Walk-forward CV splits for tuning (5 folds for speed)
def walk_forward_cv_indices(df, min_train_months=18, test_months=6, gap_months=1, n_folds=5):
    """Generate walk-forward splits, return last n_folds as list of (train, test) index arrays."""
    df = df.copy()
    df["date_key"] = df["transaction_year"] * 100 + df["transaction_month"]
    all_months = sorted(df["date_key"].unique())

    all_splits = []
    test_start_idx = min_train_months + gap_months

    while test_start_idx + test_months <= len(all_months):
        train_months_list = all_months[:test_start_idx - gap_months]
        test_months_list = all_months[test_start_idx:test_start_idx + test_months]

        train_mask = df["date_key"].isin(train_months_list)
        test_mask = df["date_key"].isin(test_months_list)

        all_splits.append((
            np.where(train_mask)[0],
            np.where(test_mask)[0]
        ))
        test_start_idx += test_months

    return all_splits[-n_folds:]


cv_splits = walk_forward_cv_indices(df_train, n_folds=5)
print(f"Walk-forward CV folds for tuning: {len(cv_splits)}")
for i, (tr, te) in enumerate(cv_splits):
    print(f"  Fold {i+1}: train {len(tr):>6} | test {len(te):>5}")

Walk-forward CV folds for tuning: 5
  Fold 1: train 595887 | test 12434
  Fold 2: train 608821 | test 13375
  Fold 3: train 621627 | test 14230
  Fold 4: train 635441 | test 13279
  Fold 5: train 649444 | test 13368


In [15]:
def compute_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "R2": r2_score(y_true, y_pred),
        "MdAE": median_absolute_error(y_true, y_pred),
        "PER10": np.mean(np.abs(y_pred - y_true) / y_true <= 0.10) * 100,
    }

## 2. Tune LightGBM

In [16]:
lgbm_param_dist = {
    "n_estimators": randint(100, 500),
    "learning_rate": uniform(0.01, 0.19),       # 0.01 to 0.20
    "num_leaves": randint(31, 127),
    "max_depth": randint(5, 15),
    "min_child_samples": randint(10, 50),
    "colsample_bytree": uniform(0.5, 0.5),       # 0.5 to 1.0
    "subsample": uniform(0.6, 0.4),              # 0.6 to 1.0
    "reg_alpha": uniform(0, 1.0),
    "reg_lambda": uniform(0, 1.0),
}

lgbm_base = LGBMRegressor(n_jobs=-1, random_state=42, verbose=-1)

print("Tuning LightGBM — 50 iterations × 5 walk-forward folds...")
start = time.time()

lgbm_search = RandomizedSearchCV(
    lgbm_base,
    param_distributions=lgbm_param_dist,
    n_iter=50,
    cv=cv_splits,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
lgbm_search.fit(X_train_t, y_train)

elapsed = time.time() - start
print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Best CV MAE: ${-lgbm_search.best_score_:,.0f}")
print(f"\nBest params:")
for k, v in lgbm_search.best_params_.items():
    print(f"  {k}: {v}")

Tuning LightGBM — 50 iterations × 5 walk-forward folds...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Done in 21.3 minutes
Best CV MAE: $29,320

Best params:
  colsample_bytree: 0.7583481287132834
  learning_rate: 0.1348511524150317
  max_depth: 9
  min_child_samples: 31
  n_estimators: 448
  num_leaves: 97
  reg_alpha: 0.24773098950115746
  reg_lambda: 0.3559726786512616
  subsample: 0.9031384441857476


## 3. Tune XGBoost

In [17]:
xgb_param_dist = {
    "n_estimators": randint(100, 500),
    "learning_rate": uniform(0.01, 0.19),
    "max_depth": randint(4, 12),
    "min_child_weight": randint(1, 10),
    "colsample_bytree": uniform(0.5, 0.5),
    "subsample": uniform(0.6, 0.4),
    "gamma": uniform(0, 0.5),
    "reg_alpha": uniform(0, 1.0),
    "reg_lambda": uniform(0, 1.0),
}

xgb_base = XGBRegressor(n_jobs=-1, random_state=42, verbosity=0)

print("Tuning XGBoost — 50 iterations × 5 walk-forward folds...")
start = time.time()

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=50,
    cv=cv_splits,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
xgb_search.fit(X_train_t, y_train)

elapsed = time.time() - start
print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Best CV MAE: ${-xgb_search.best_score_:,.0f}")
print(f"\nBest params:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k}: {v}")

Tuning XGBoost — 50 iterations × 5 walk-forward folds...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Done in 93.4 minutes
Best CV MAE: $29,473

Best params:
  colsample_bytree: 0.816550728636634
  gamma: 0.16951489552435034
  learning_rate: 0.07634981917640557
  max_depth: 11
  min_child_weight: 3
  n_estimators: 251
  reg_alpha: 0.4045081271221901
  reg_lambda: 0.8877700987609598
  subsample: 0.9403713795070051


## 4. Evaluate Tuned Models on Holdout Test Set

In [18]:
# Tuned models are already refit on full training data by RandomizedSearchCV
lgbm_tuned = lgbm_search.best_estimator_
xgb_tuned = xgb_search.best_estimator_

# Predict on holdout
y_pred_lgbm = lgbm_tuned.predict(X_test_t)
y_pred_xgb = xgb_tuned.predict(X_test_t)
y_pred_ensemble = (y_pred_lgbm + y_pred_xgb) / 2

lgbm_metrics = compute_metrics(y_test, y_pred_lgbm)
xgb_metrics = compute_metrics(y_test, y_pred_xgb)
ensemble_metrics = compute_metrics(y_test, y_pred_ensemble)

print("=" * 80)
print("TUNED MODEL RESULTS — Holdout Test Set")
print("=" * 80)

for name, m in [("LightGBM (tuned)", lgbm_metrics),
                ("XGBoost (tuned)", xgb_metrics),
                ("Ensemble (tuned)", ensemble_metrics)]:
    print(f"\n{name}:")
    print(f"  MAE: ${m['MAE']:,.0f} | RMSE: ${m['RMSE']:,.0f} | "
          f"MAPE: {m['MAPE']:.1f}% | R²: {m['R2']:.4f} | "
          f"MdAE: ${m['MdAE']:,.0f} | PER10: {m['PER10']:.1f}%")

TUNED MODEL RESULTS — Holdout Test Set

LightGBM (tuned):
  MAE: $25,565 | RMSE: $36,617 | MAPE: 3.9% | R²: 0.9702 | MdAE: $18,562 | PER10: 94.6%

XGBoost (tuned):
  MAE: $24,609 | RMSE: $35,458 | MAPE: 3.7% | R²: 0.9721 | MdAE: $17,716 | PER10: 94.8%

Ensemble (tuned):
  MAE: $24,476 | RMSE: $35,234 | MAPE: 3.7% | R²: 0.9724 | MdAE: $17,674 | PER10: 95.1%


## 5. Compare Default vs Tuned

In [19]:
comparison = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "MAPE", "R²", "MdAE", "PER10"],
    "Default Ensemble": [
        "$26,044", "$37,174", "3.9%", "0.9693", "$18,881", "94.2%"
    ],
    "Tuned Ensemble": [
        f"${ensemble_metrics['MAE']:,.0f}",
        f"${ensemble_metrics['RMSE']:,.0f}",
        f"{ensemble_metrics['MAPE']:.1f}%",
        f"{ensemble_metrics['R2']:.4f}",
        f"${ensemble_metrics['MdAE']:,.0f}",
        f"{ensemble_metrics['PER10']:.1f}%",
    ],
}).set_index("Metric")

default_mae = 26044
tuned_mae = ensemble_metrics["MAE"]
improvement = default_mae - tuned_mae

print("=" * 60)
print("DEFAULT vs TUNED COMPARISON")
print("=" * 60)
print(comparison.to_string())
print(f"\nMAE improvement: ${improvement:,.0f} ({improvement/default_mae*100:.1f}%)")
if improvement > 0:
    print("Tuning improved the model")
else:
    print("Tuning did not improve — default params were already good")

DEFAULT vs TUNED COMPARISON
       Default Ensemble Tuned Ensemble
Metric                                
MAE             $26,044        $24,476
RMSE            $37,174        $35,234
MAPE               3.9%           3.7%
R²               0.9693         0.9724
MdAE            $18,881        $17,674
PER10             94.2%          95.1%

MAE improvement: $1,568 (6.0%)
Tuning improved the model


## 6. Save Tuned Models (if improved)

In [20]:
MODEL_VERSION = "v3_macro_tuned"

if improvement > 0:
    version_dir = f"../models/{MODEL_VERSION}"
    os.makedirs(version_dir, exist_ok=True)
    joblib.dump(lgbm_tuned, f"{version_dir}/lgbm_model.joblib")
    joblib.dump(xgb_tuned, f"{version_dir}/xgb_model.joblib")
    joblib.dump(preprocessor, f"{version_dir}/preprocessor.joblib")

    best_params = {
        "lgbm": lgbm_search.best_params_,
        "xgb": xgb_search.best_params_,
        "lgbm_cv_mae": float(-lgbm_search.best_score_),
        "xgb_cv_mae": float(-xgb_search.best_score_),
        "ensemble_holdout_mae": float(ensemble_metrics["MAE"]),
        "version": MODEL_VERSION,
    }
    for model_key in ["lgbm", "xgb"]:
        for k, v in best_params[model_key].items():
            if hasattr(v, "item"):
                best_params[model_key][k] = v.item()

    with open(f"{version_dir}/best_params.json", "w") as f:
        json.dump(best_params, f, indent=2)

    joblib.dump(lgbm_tuned, "../models/lgbm_model.joblib")
    joblib.dump(xgb_tuned, "../models/xgb_model.joblib")
    joblib.dump(preprocessor, "../models/preprocessor.joblib")
    with open("../models/best_params.json", "w") as f:
        json.dump(best_params, f, indent=2)

    print(f"Saved to ../models/{MODEL_VERSION}/:")
    for f in sorted(os.listdir(version_dir)):
        size_kb = os.path.getsize(f"{version_dir}/{f}") / 1024
        print(f"  {f:30s} {size_kb:.0f} KB")
    print(f"\nAlso copied to ../models/ (latest)")
else:
    print("Keeping previous models — no improvement from tuning.")

Saved to ../models/v3_macro_tuned/:
  best_params.json               1 KB
  lgbm_model.joblib              3834 KB
  preprocessor.joblib            8 KB
  xgb_model.joblib               16422 KB

Also copied to ../models/ (latest)
